In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [15]:
from collections import Counter
import math

In [2]:
df = pd.read_parquet("20240607.parquet")

print(df.shape)
df.head()

(82408, 37)


,DST_IP,DST_IP_SUBNET,DST_IP_VERSION,DST_ASN,DST_COUNTRY,DST_PORT,PROTOCOL,TIME_FIRST,TIME_LAST,DURATION,...,QUIC_TLS_EXT_TYPE,QUIC_PACKETS,PPI,PPI_LEN,PPI_DURATION,PPI_ROUNDTRIPS,PHIST_SRC_SIZES,PHIST_DST_SIZES,PHIST_SRC_IPT,PHIST_DST_IPT
0,2b8f7601cb,0415094b82,4,13335,ZZ,443,17,2024-06-06 23:00:00+02:00,2024-06-06 23:00:00.200524+02:00,0.200524,...,"[51, 16, 13, 27, 42, 10, 0, 57, 65037, 45, 175...","[129, 130, 133, 128, 128, 128, 128, 132, 128, ...","[[0, 0, 2, 0, 0, 0, 0, 4, 0, 0, 0, 0, 1, 0, 0,...",21,0.200,4,"[0, 0, 6, 2, 0, 0, 1, 2]","[0, 6, 2, 0, 0, 0, 1, 1]","[7, 0, 2, 1, 0, 0, 0, 0]","[7, 0, 1, 1, 0, 0, 0, 0]"
1,3f9660e9b2,38f424d20a,4,20940,AT,443,17,2024-06-06 23:00:00+02:00,2024-06-06 23:01:54.420675+02:00,114.420675,...,"[51, 43, 13, 17513, 27, 45, 0, 65445, 16, 10, 41]","[129, 129, 132, 129, 132, 132, 128, 128, 128, ...","[[0, 6, 0, 18, 1, 1, 0, 5, 0, 0, 6, 0, 0, 0, 0...",30,0.044,3,"[0, 0, 56, 0, 1, 5, 2, 2]","[0, 21, 0, 2, 0, 2, 9, 79]","[48, 5, 1, 2, 1, 0, 0, 8]","[93, 5, 2, 3, 1, 0, 0, 8]"
2,e5d823e4a9,1ee05db139,4,32934,CZ,443,17,2024-06-06 23:00:00+02:00,2024-06-06 23:00:59.986355+02:00,59.986355,...,"[43, 10, 51, 13, 0, 16, 45, 65445, 41]","[129, 129, 132, 128, 129, 132, 128, 128, 128, ...","[[0, 1, 0, 1, 6, 1, 0, 0, 0, 0, 0, 1, 0, 7, 27...",22,59.986,5,"[0, 0, 4, 1, 0, 1, 0, 5]","[0, 0, 4, 2, 1, 1, 2, 1]","[8, 0, 0, 1, 0, 0, 0, 1]","[7, 0, 2, 0, 0, 0, 0, 1]"
3,9579089b90,1ee05db139,4,32934,CZ,443,17,2024-06-06 23:00:00+02:00,2024-06-06 23:00:02.939166+02:00,2.939166,...,"[43, 10, 51, 13, 0, 16, 45, 42, 27, 65445, 41]","[129, 130, 130, 129, 132, 128, 128, 128, 128, ...","[[0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 7, 1, 0, 0, 1,...",30,0.202,7,"[0, 0, 37, 1, 0, 14, 0, 3]","[0, 0, 30, 2, 11, 8, 9, 16]","[40, 5, 2, 2, 4, 0, 0, 1]","[57, 9, 2, 5, 1, 0, 0, 1]"
4,282c465d80,6959317fea,4,13335,ZZ,443,17,2024-06-06 23:00:00+02:00,2024-06-06 23:00:00.122451+02:00,0.122451,...,"[65037, 45, 57, 13, 10, 27, 16, 0, 51, 43, 175...","[129, 133, 132, 128, 128, 128, 128, 128, 128, ...","[[0, 1, 5, 1, 0, 0, 0, 0, 1, 3, 0, 1, 32, 6, 3...",17,0.123,4,"[0, 0, 5, 0, 1, 0, 1, 1]","[0, 4, 1, 0, 0, 1, 2, 1]","[4, 0, 3, 0, 0, 0, 0, 0]","[6, 0, 2, 0, 0, 0, 0, 0]"


In [3]:
ppi_features = df[
    ["PPI",
     "PPI_LEN",
     "PPI_DURATION",
     "PPI_ROUNDTRIPS"]].copy()

ppi_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 82408 entries, 0 to 82407
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PPI             82408 non-null  object 
 1   PPI_LEN         82408 non-null  int64  
 2   PPI_DURATION    82408 non-null  float64
 3   PPI_ROUNDTRIPS  82408 non-null  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 2.5+ MB


In [4]:
# Calculate the average inter-packet time for each flow
ppi_features["mean_ipt"] = ppi_features["PPI"].apply(
    lambda x: np.mean(x[0])
)

In [5]:
# Calculate the variability of inter-packet times
ppi_features["std_ipt"] = ppi_features["PPI"].apply(
    lambda x: np.std(x[0])
)

In [6]:
# Calculate the average packet size
ppi_features["mean_packet_size"] = ppi_features["PPI"].apply(
    lambda x: np.mean(x[2])
)

In [7]:
# Calculate packet size variability
ppi_features["std_packet_size"] = ppi_features["PPI"].apply(
    lambda x: np.std(x[2])
)

In [8]:
def direction_change_ratio(directions):

    # If there is only one packet, no direction change is possible
    if len(directions) < 2:
        return 0

    changes = 0

    # Compare every packet direction with the next one
    for i in range(len(directions) - 1):

        if directions[i] != directions[i + 1]:
            changes += 1

    # Return the proportion of direction changes
    return changes / (len(directions) - 1)

In [9]:
ppi_features["direction_change_ratio"] = (
    ppi_features["PPI"].apply(
        lambda x: direction_change_ratio(x[1])
    )
)

In [10]:
def forward_packet_ratio(directions):

    if len(directions) == 0:
        return 0

    forward_packets = np.sum(directions == 1)

    return forward_packets / len(directions)

In [11]:
ppi_features["forward_packet_ratio"] = (
    ppi_features["PPI"].apply(
        lambda x: forward_packet_ratio(x[1])
    )
)

In [12]:
ppi_features["log_ppi_duration"] = np.log1p(ppi_features["PPI_DURATION"])
ppi_features["log_mean_ipt"] = np.log1p(ppi_features["mean_ipt"])
ppi_features["log_std_ipt"] = np.log1p(ppi_features["std_ipt"])

In [13]:
ppi_final = ppi_features[
    [
        # Original Features
        "PPI_LEN",
        "PPI_ROUNDTRIPS",

        # Log-transformed Features
        "log_ppi_duration",
        "log_mean_ipt",
        "log_std_ipt",

        # Engineered Features
        "mean_packet_size",
        "std_packet_size",
        "direction_change_ratio",
        "forward_packet_ratio"
    ]
].copy()

print("Final PPI Feature Bank Shape:", ppi_final.shape)

ppi_final.head()

Final PPI Feature Bank Shape: (82408, 9)


,PPI_LEN,PPI_ROUNDTRIPS,log_ppi_duration,log_mean_ipt,log_std_ipt,mean_packet_size,std_packet_size,direction_change_ratio,forward_packet_ratio
0,21,4,0.182322,2.353640,3.109561,278.523810,441.632968,0.400000,0.523810
1,30,3,0.043059,0.902868,1.504042,1011.666667,510.923434,0.172414,0.200000
2,22,5,4.110644,7.911191,9.431400,460.636364,500.916845,0.428571,0.500000
3,30,7,0.183987,2.045540,3.163525,384.866667,454.480343,0.448276,0.400000
4,17,4,0.116004,2.108429,2.607685,333.882353,421.149252,0.500000,0.470588


In [14]:
cid_features = df[
    [
        "DST_ASN",
        "QUIC_OCCID",
        "QUIC_OSCID",
        "QUIC_SCID",
        "QUIC_RETRY_SCID",
        "QUIC_SNI"
    ]
].copy()

cid_features.head()

,DST_ASN,QUIC_OCCID,QUIC_OSCID,QUIC_SCID,QUIC_RETRY_SCID,QUIC_SNI
0,13335,,f348c3bf69470769,01fe92ee8f4feee65dfe8deede4fc3db7ac84d6c,,discord.com
1,20940,,ca6f3ae88aa61dae,0ca8ea9c66805c2d,,p16-sign-va.tiktokcdn.com
2,32934,,e495eaf9c3c3df22,ac21000a1bcfe577,,graph.facebook.com
3,32934,,de3bbc6f2bc2575b,8c240031120c534d,,scontent-prg1-1.xx.fbcdn.net
4,13335,,8aaad29c59a49a64,01dc1bd590338e21aadc04d50a33847c5ae1f740,,exp.notion.so


In [16]:
cid_features["occid_length"] = cid_features["QUIC_OCCID"].str.len()

cid_features["oscid_length"] = cid_features["QUIC_OSCID"].str.len()

cid_features["scid_length"] = cid_features["QUIC_SCID"].str.len()

cid_features["retry_length"] = cid_features["QUIC_RETRY_SCID"].str.len()

cid_features["sni_length"] = cid_features["QUIC_SNI"].str.len()

cid_features["sni_labels"] = (
    cid_features["QUIC_SNI"]
    .str.split(".")
    .str.len()
)

In [17]:
cid_features["retry_present"] = (cid_features["retry_length"] > 0).astype(int)

In [18]:
def shannon_entropy(text):

    if len(text) == 0:
        return 0.0

    counts = Counter(text)

    entropy = 0.0

    length = len(text)

    for count in counts.values():

        p = count / length

        entropy -= p * math.log2(p)

    return entropy

In [19]:
def normalized_entropy(text):

    if len(text) == 0:
        return 0.0

    H = shannon_entropy(text)

    Hmax = math.log2(min(len(text), 16))

    return H / Hmax if Hmax > 0 else 0.0

In [20]:
cid_features["occid_entropy"] = (cid_features["QUIC_OCCID"].apply(normalized_entropy))

cid_features["oscid_entropy"] = (cid_features["QUIC_OSCID"].apply(normalized_entropy))

cid_features["scid_entropy"] = (cid_features["QUIC_SCID"].apply(normalized_entropy))

In [21]:
cid_features[
    [
        "occid_entropy",
        "oscid_entropy",
        "scid_entropy"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
occid_entropy,82408.0,0.162327,0.342177,0.000000,0.000000,0.000000,0.00000,1.000000
oscid_entropy,82408.0,0.810646,0.057011,0.522979,0.769455,0.812500,0.84375,0.982307
scid_entropy,82408.0,0.806787,0.087642,0.000000,0.769455,0.800705,0.85141,1.000000


In [22]:
MIN_FLOWS = max(50, int(0.001 * len(cid_features)))

In [23]:
asn_groups = cid_features.groupby("DST_ASN")

In [24]:
asn_profile = asn_groups.agg({

    "scid_length": ["count", "median"],

    "scid_entropy": "median",

    "oscid_length": "median",

    "oscid_entropy": "median",

    "retry_present": "mean",

    "sni_length": "median",

    "sni_labels": "median"

})

In [25]:
asn_profile.columns = [

    "flows",
    "median_scid_length",
    "median_scid_entropy",
    "median_oscid_length",
    "median_oscid_entropy",
    "retry_rate",
    "median_sni_length",
    "median_sni_labels",

]

In [26]:
asn_profile = asn_profile[asn_profile["flows"] >= MIN_FLOWS]

In [27]:
asn_profile.head()

,flows,median_scid_length,median_scid_entropy,median_oscid_length,median_oscid_entropy,retry_rate,median_sni_length,median_sni_labels
DST_ASN,,,,,,,,
2852,98,16.0,0.800705,16.0,0.812500,0.000000,10.0,3.0
6185,754,40.0,0.926419,16.0,0.805506,0.000000,17.0,3.0
8075,1640,28.0,0.884787,16.0,0.800705,0.002439,21.0,3.0
13335,4318,40.0,0.906168,16.0,0.812500,0.000000,14.0,3.0
15169,45800,16.0,0.800705,16.0,0.812500,0.000000,20.0,3.0


In [28]:
cid_features = cid_features.merge(
    asn_profile,
    on="DST_ASN",
    how="left"
)

In [29]:
cid_features["known_asn"] = (
    cid_features["flows"].notna().astype(int)
)

In [30]:
global_scid_length = cid_features["scid_length"].median()

global_scid_entropy = cid_features["scid_entropy"].median()

global_oscid_length = cid_features["oscid_length"].median()

global_oscid_entropy = cid_features["oscid_entropy"].median()

global_retry_rate = cid_features["retry_present"].mean()

global_sni_length = cid_features["sni_length"].median()

global_sni_labels = cid_features["sni_labels"].median()

In [31]:
fill_values = {
    "median_scid_length": global_scid_length,
    "median_scid_entropy": global_scid_entropy,
    "median_oscid_length": global_oscid_length,
    "median_oscid_entropy": global_oscid_entropy,
    "retry_rate": global_retry_rate,
    "median_sni_length": global_sni_length,
    "median_sni_labels": global_sni_labels
}

cid_features = cid_features.fillna(fill_values)

In [32]:
asn_profile.describe().T

,count,mean,std,min,25%,50%,75%,max
flows,15.0,5460.266667,11913.592276,98.000000,548.500000,1414.000000,3351.000000,45800.000000
median_scid_length,15.0,26.533333,11.173097,16.000000,16.000000,24.000000,40.000000,40.000000
median_scid_entropy,15.0,0.854054,0.062918,0.769455,0.800705,0.864787,0.917386,0.926419
median_oscid_length,15.0,16.533333,2.065591,16.000000,16.000000,16.000000,16.000000,24.000000
median_oscid_entropy,15.0,0.816514,0.018624,0.800705,0.803105,0.812500,0.820160,0.872650
retry_rate,15.0,0.000163,0.000630,0.000000,0.000000,0.000000,0.000000,0.002439
median_sni_length,15.0,18.933333,6.134989,10.000000,15.000000,18.000000,20.500000,33.000000
median_sni_labels,15.0,3.066667,0.258199,3.000000,3.000000,3.000000,3.000000,4.000000


In [33]:
asn_profile.sort_values("flows", ascending=False).head(20)

,flows,median_scid_length,median_scid_entropy,median_oscid_length,median_oscid_entropy,retry_rate,median_sni_length,median_sni_labels
DST_ASN,,,,,,,,
15169,45800,16.0,0.800705,16.0,0.812500,0.000000,20.0,3.0
32934,16848,16.0,0.769455,16.0,0.800705,0.000000,18.0,3.0
13335,4318,40.0,0.906168,16.0,0.812500,0.000000,14.0,3.0
396982,4195,16.0,0.800705,16.0,0.800705,0.000000,29.0,3.0
36183,2507,40.0,0.925925,16.0,0.812500,0.000000,15.0,3.0
20940,1670,16.0,0.788910,16.0,0.812500,0.000000,25.0,3.0
8075,1640,28.0,0.884787,16.0,0.800705,0.002439,21.0,3.0
16509,1414,40.0,0.921207,16.0,0.820160,0.000000,16.0,3.0
396986,1097,16.0,0.800705,16.0,0.800705,0.000000,33.0,3.0


In [34]:
cid_features["scid_length_deviation"] = (
    cid_features["scid_length"] -
    cid_features["median_scid_length"]
).abs()

cid_features["oscid_length_deviation"] = (
    cid_features["oscid_length"] -
    cid_features["median_oscid_length"]
).abs()

In [35]:
cid_features["sni_length_deviation"] = (
    cid_features["sni_length"] -
    cid_features["median_sni_length"]
).abs()

In [36]:
cid_features["scid_entropy_deviation"] = (
    cid_features["scid_entropy"] -
    cid_features["median_scid_entropy"]
).abs()

cid_features["oscid_entropy_deviation"] = (
    cid_features["oscid_entropy"] -
    cid_features["median_oscid_entropy"]
).abs()

In [41]:
cid_final = cid_features[
    [
        # Raw CID Features
        "occid_length",
        "oscid_length",
        "scid_length",
        "retry_present",
        "sni_length",
        "sni_labels",

        # Statistical Features
        "oscid_entropy",
        "scid_entropy",

        # Context Features
        "scid_length_deviation",
        "oscid_length_deviation",
        "scid_entropy_deviation",
        "oscid_entropy_deviation",
        "sni_length_deviation",
        "known_asn"
    ]
].copy()


print(cid_final.shape)

(82408, 14)


In [40]:
cid_final.head()

,occid_length,oscid_length,scid_length,retry_present,sni_length,sni_labels,oscid_entropy,scid_entropy,scid_length_deviation,oscid_length_deviation,scid_entropy_deviation,oscid_entropy_deviation,sni_length_deviation,known_asn
0,0,16,40,0,11,2,0.812500,0.876087,0.0,0.0,0.030081,0.000000,3.0,1
1,0,16,16,0,25,3,0.724849,0.800705,0.0,0.0,0.011795,0.087651,0.0,1
2,0,16,16,0,18,3,0.812500,0.800705,0.0,0.0,0.031250,0.011795,0.0,1
3,0,16,16,0,28,4,0.781250,0.769455,0.0,0.0,0.000000,0.019455,10.0,1
4,0,16,40,0,13,3,0.713054,0.931268,0.0,0.0,0.025100,0.099446,1.0,1


In [42]:
hist_features = df[
    [
        "PHIST_SRC_SIZES",
        "PHIST_DST_SIZES",
        "PHIST_SRC_IPT",
        "PHIST_DST_IPT"
    ]
].copy()

In [43]:
def histogram_entropy(hist):

    # Convert to NumPy array
    hist = np.array(hist, dtype=float)

    # Total observations
    total = hist.sum()

    # Handle empty histograms
    if total == 0:
        return 0

    # Convert counts to probabilities
    probabilities = hist / total

    # Calculate Shannon entropy
    entropy = 0

    for p in probabilities:
        if p > 0:
            entropy -= p * math.log2(p)

    # Normalize entropy
    max_entropy = math.log2(len(hist))

    return entropy / max_entropy

In [44]:
# Source packet size histogram entropy
hist_features["src_size_entropy"] = (
    hist_features["PHIST_SRC_SIZES"]
    .apply(histogram_entropy)
)

# Destination packet size histogram entropy
hist_features["dst_size_entropy"] = (
    hist_features["PHIST_DST_SIZES"]
    .apply(histogram_entropy)
)

# Source IPT histogram entropy
hist_features["src_ipt_entropy"] = (
    hist_features["PHIST_SRC_IPT"]
    .apply(histogram_entropy)
)

# Destination IPT histogram entropy
hist_features["dst_ipt_entropy"] = (
    hist_features["PHIST_DST_IPT"]
    .apply(histogram_entropy)
)

In [45]:
hist_features[
    [
        "src_size_entropy",
        "dst_size_entropy",
        "src_ipt_entropy",
        "dst_ipt_entropy"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
src_size_entropy,82408.0,0.557661,0.184561,0.0,0.491825,0.597494,0.685012,0.913191
dst_size_entropy,82408.0,0.574623,0.210322,0.0,0.513245,0.639432,0.709362,0.935785
src_ipt_entropy,82408.0,0.351961,0.189649,0.0,0.226136,0.335095,0.483489,0.964737
dst_ipt_entropy,82408.0,0.341818,0.211914,0.0,0.181188,0.329394,0.497038,0.977070


In [46]:
hist_final = hist_features[
    [
        "src_size_entropy",
        "dst_size_entropy",
        "src_ipt_entropy",
        "dst_ipt_entropy"
    ]
].copy()

print(hist_final.shape)

hist_final.head()

(82408, 4)


,src_size_entropy,dst_size_entropy,src_ipt_entropy,dst_ipt_entropy
0,0.561939,0.523650,0.385593,0.328809
1,0.293477,0.436323,0.439826,0.333088
2,0.558912,0.789508,0.307309,0.385593
3,0.407074,0.750231,0.458501,0.411323
4,0.516265,0.686271,0.328409,0.270426


In [47]:
flow_df = df.copy()

In [48]:
flow_df["duration_safe"] = flow_df["DURATION"].clip(lower=1e-6)

In [49]:
flow_df["total_bytes"] = (
    flow_df["BYTES"] +
    flow_df["BYTES_REV"]
)

In [50]:
flow_df["total_packets"] = (
    flow_df["PACKETS"] +
    flow_df["PACKETS_REV"]
)

In [51]:
flow_df["byte_rate"] = (
    flow_df["total_bytes"] /
    flow_df["duration_safe"]
)

In [52]:
flow_df["packet_rate"] = (
    flow_df["total_packets"] /
    flow_df["duration_safe"]
)

In [53]:
flow_df["avg_packet_size"] = (
    flow_df["total_bytes"] /
    flow_df["total_packets"].clip(lower=1)
)

In [54]:
flow_df["byte_ratio"] = (
    flow_df["BYTES"] /
    flow_df["BYTES_REV"].clip(lower=1)
)

In [55]:
flow_df["packet_ratio"] = (
    flow_df["PACKETS"] /
    flow_df["PACKETS_REV"].clip(lower=1)
)

In [56]:
flow_features = flow_df[
    [
        "DURATION",
        "FLOW_END_REASON",
        "total_bytes",
        "total_packets",
        "byte_rate",
        "packet_rate",
        "avg_packet_size",
        "byte_ratio",
        "packet_ratio"
    ]
]

In [57]:
flow_features.replace([np.inf, -np.inf], np.nan, inplace=True)

flow_features.isnull().sum()

DURATION           0
FLOW_END_REASON    0
total_bytes        0
total_packets      0
byte_rate          0
packet_rate        0
avg_packet_size    0
byte_ratio         0
packet_ratio       0
dtype: int64

In [58]:
flow_features = flow_df[
    [
        "DURATION",
        "FLOW_END_REASON",
        "total_bytes",
        "total_packets",
        "byte_rate",
        "packet_rate",
        "avg_packet_size",
        "byte_ratio",
        "packet_ratio"
    ]
].copy()

In [59]:
flow_features["log_duration"] = np.log1p(flow_features["DURATION"])

flow_features["log_total_bytes"] = np.log1p(flow_features["total_bytes"])

flow_features["log_total_packets"] = np.log1p(flow_features["total_packets"])

flow_features["log_byte_rate"] = np.log1p(flow_features["byte_rate"])

flow_features["log_packet_rate"] = np.log1p(flow_features["packet_rate"])

flow_features["log_byte_ratio"] = np.log1p(flow_features["byte_ratio"])

flow_features["log_packet_ratio"] = np.log1p(flow_features["packet_ratio"])

In [60]:
log_columns = [

"log_duration",

"FLOW_END_REASON",

"log_total_bytes",

"log_total_packets",

"log_byte_rate",

"log_packet_rate",

"avg_packet_size",

"log_byte_ratio",

"log_packet_ratio"

]

flow_features[log_columns].corr()

,log_duration,FLOW_END_REASON,log_total_bytes,log_total_packets,log_byte_rate,log_packet_rate,avg_packet_size,log_byte_ratio,log_packet_ratio
log_duration,1.000000,0.182477,0.501267,0.568937,-0.784465,-0.816614,0.037921,-0.001629,-0.121771
FLOW_END_REASON,0.182477,1.000000,0.097922,0.116347,-0.102979,-0.094824,-0.003846,-0.003741,-0.012182
log_total_bytes,0.501267,0.097922,1.000000,0.971072,0.033748,-0.070339,0.527831,-0.234859,-0.402031
log_total_packets,0.568937,0.116347,0.971072,1.000000,-0.075691,-0.144439,0.326023,-0.195566,-0.345319
log_byte_rate,-0.784465,-0.102979,0.033748,-0.075691,1.000000,0.979459,0.377508,-0.135278,-0.136757
log_packet_rate,-0.816614,-0.094824,-0.070339,-0.144439,0.979459,1.000000,0.218631,-0.098409,-0.072186
avg_packet_size,0.037921,-0.003846,0.527831,0.326023,0.377508,0.218631,1.000000,-0.223035,-0.387709
log_byte_ratio,-0.001629,-0.003741,-0.234859,-0.195566,-0.135278,-0.098409,-0.223035,1.000000,0.727086
log_packet_ratio,-0.121771,-0.012182,-0.402031,-0.345319,-0.136757,-0.072186,-0.387709,0.727086,1.000000


In [61]:
selected_columns = [

    "log_duration",
    "FLOW_END_REASON",
    "log_total_bytes",
    "log_byte_rate",
    "avg_packet_size",
    "log_byte_ratio",
    "log_packet_ratio"

]

flow_features_selected = flow_features[selected_columns].copy()

flow_features_selected.head()

,log_duration,FLOW_END_REASON,log_total_bytes,log_byte_rate,avg_packet_size,log_byte_ratio,log_packet_ratio
0,0.182758,1,8.769973,10.376670,306.523810,1.025248,0.741937
1,4.748583,1,11.735229,6.996255,697.731844,0.080153,0.459998
2,4.110650,1,9.282754,5.194109,488.636364,1.098147,0.693147
3,1.370969,1,10.756348,9.678264,358.213740,0.293910,0.544464
4,0.115515,1,8.724695,10.824597,361.882353,0.585384,0.635989


In [62]:
quic_features = df[
    [
        "QUIC_VERSION",
        "QUIC_CLIENT_VERSION",
        "QUIC_TOKEN_LENGTH",
        "QUIC_ZERO_RTT",
        "QUIC_MULTIPLEXED"
    ]
].copy()

quic_features.head()

,QUIC_VERSION,QUIC_CLIENT_VERSION,QUIC_TOKEN_LENGTH,QUIC_ZERO_RTT,QUIC_MULTIPLEXED
0,1,1,0,1,0
1,4278190109,4278190109,0,0,0
2,4207849474,4207849474,0,0,0
3,4207849474,4207849474,60,2,0
4,1,1,0,0,0


In [63]:
quic_features["token_present"] = (
    quic_features["QUIC_TOKEN_LENGTH"] > 0
).astype(int)

In [64]:
quic_features["log_token_length"] = np.log1p(
    quic_features["QUIC_TOKEN_LENGTH"]
)

In [65]:
quic_features["zero_rtt_present"] = (
    quic_features["QUIC_ZERO_RTT"] > 0
).astype(int)

In [66]:
quic_features["multiplexed"] = (
    quic_features["QUIC_MULTIPLEXED"] > 0
).astype(int)

In [67]:
version_map = {
    version: idx
    for idx, version in enumerate(
        sorted(quic_features["QUIC_VERSION"].unique())
    )
}

quic_features["quic_version_id"] = (
    quic_features["QUIC_VERSION"].map(version_map)
)

In [68]:
version_map

{np.int64(1): 0,
 np.int64(3467641594): 1,
 np.int64(4207849474): 2,
 np.int64(4207849486): 3,
 np.int64(4207849491): 4,
 np.int64(4278190109): 5}

In [69]:
quic_features["log_zero_rtt"] = np.log1p(
    quic_features["QUIC_ZERO_RTT"]
)

In [70]:
quic_features["log_multiplexed"] = np.log1p(
    quic_features["QUIC_MULTIPLEXED"]
)

In [71]:
quic_selected = quic_features[
    [
        "quic_version_id",
        "log_token_length",
        "log_zero_rtt",
        "zero_rtt_present",
        "log_multiplexed",
        "multiplexed"
    ]
]

quic_selected.corr()

,quic_version_id,log_token_length,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
quic_version_id,1.000000,0.103983,-0.025564,-0.030198,-0.052186,-0.055860
log_token_length,0.103983,1.000000,0.622754,0.682939,-0.002857,0.000260
log_zero_rtt,-0.025564,0.622754,1.000000,0.891172,-0.051570,-0.067655
zero_rtt_present,-0.030198,0.682939,0.891172,1.000000,-0.074695,-0.086205
log_multiplexed,-0.052186,-0.002857,-0.051570,-0.074695,1.000000,0.929812
multiplexed,-0.055860,0.000260,-0.067655,-0.086205,0.929812,1.000000


In [72]:
selected_quic = quic_features[
    [
        "quic_version_id",
        "QUIC_TOKEN_LENGTH",
        "log_zero_rtt",
        "zero_rtt_present",
        "log_multiplexed",
        "multiplexed"
    ]
].copy()

selected_quic.head()

,quic_version_id,QUIC_TOKEN_LENGTH,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
0,0,0,0.693147,1,0.0,0
1,5,0,0.000000,0,0.0,0
2,2,0,0.000000,0,0.0,0
3,2,60,1.098612,1,0.0,0
4,0,0,0.000000,0,0.0,0


In [75]:
with_CID_df = pd.concat([ppi_final,cid_final,hist_final,flow_features_selected,selected_quic],axis=1)

print(with_CID_df.shape)

with_CID_df.to_parquet(
    "with_CID.parquet",
    index=False
)

(82408, 40)


In [76]:
check_df = pd.read_parquet("with_CID.parquet")

print(check_df.shape)
check_df.head()

(82408, 40)


,PPI_LEN,PPI_ROUNDTRIPS,log_ppi_duration,log_mean_ipt,log_std_ipt,mean_packet_size,std_packet_size,direction_change_ratio,forward_packet_ratio,occid_length,...,log_byte_rate,avg_packet_size,log_byte_ratio,log_packet_ratio,quic_version_id,QUIC_TOKEN_LENGTH,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
0,21,4,0.182322,2.353640,3.109561,278.523810,441.632968,0.400000,0.523810,0,...,10.376670,306.523810,1.025248,0.741937,0,0,0.693147,1,0.0,0
1,30,3,0.043059,0.902868,1.504042,1011.666667,510.923434,0.172414,0.200000,0,...,6.996255,697.731844,0.080153,0.459998,5,0,0.000000,0,0.0,0
2,22,5,4.110644,7.911191,9.431400,460.636364,500.916845,0.428571,0.500000,0,...,5.194109,488.636364,1.098147,0.693147,2,0,0.000000,0,0.0,0
3,30,7,0.183987,2.045540,3.163525,384.866667,454.480343,0.448276,0.400000,0,...,9.678264,358.213740,0.293910,0.544464,2,60,1.098612,1,0.0,0
4,17,4,0.116004,2.108429,2.607685,333.882353,421.149252,0.500000,0.470588,0,...,10.824597,361.882353,0.585384,0.635989,0,0,0.000000,0,0.0,0


In [77]:
without_CID_df = pd.concat([ppi_final,hist_final,flow_features_selected,selected_quic],axis=1)

print(without_CID_df.shape)

without_CID_df.to_parquet(
    "without_CID.parquet",
    index=False
)

(82408, 26)


In [78]:
check_df2 = pd.read_parquet("without_CID.parquet")

print(check_df2.shape)
check_df2.head()

(82408, 26)


,PPI_LEN,PPI_ROUNDTRIPS,log_ppi_duration,log_mean_ipt,log_std_ipt,mean_packet_size,std_packet_size,direction_change_ratio,forward_packet_ratio,src_size_entropy,...,log_byte_rate,avg_packet_size,log_byte_ratio,log_packet_ratio,quic_version_id,QUIC_TOKEN_LENGTH,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
0,21,4,0.182322,2.353640,3.109561,278.523810,441.632968,0.400000,0.523810,0.561939,...,10.376670,306.523810,1.025248,0.741937,0,0,0.693147,1,0.0,0
1,30,3,0.043059,0.902868,1.504042,1011.666667,510.923434,0.172414,0.200000,0.293477,...,6.996255,697.731844,0.080153,0.459998,5,0,0.000000,0,0.0,0
2,22,5,4.110644,7.911191,9.431400,460.636364,500.916845,0.428571,0.500000,0.558912,...,5.194109,488.636364,1.098147,0.693147,2,0,0.000000,0,0.0,0
3,30,7,0.183987,2.045540,3.163525,384.866667,454.480343,0.448276,0.400000,0.407074,...,9.678264,358.213740,0.293910,0.544464,2,60,1.098612,1,0.0,0
4,17,4,0.116004,2.108429,2.607685,333.882353,421.149252,0.500000,0.470588,0.516265,...,10.824597,361.882353,0.585384,0.635989,0,0,0.000000,0,0.0,0


In [79]:
print(with_CID_df.columns.duplicated().sum())
print(without_CID_df.columns.duplicated().sum())

0
0
